In [1]:
import math
import pandas as pd
import numpy as np
import random
import os
import string


In [2]:
!pip install pandas

# Part1  Utility Functions (From Scratch)

In [3]:

def compute_mean(values):
    total = 0.0
    n = len(values)
    for v in values:
        total += v
    return total / n


def compute_variance(values):
    mu = compute_mean(values)
    total = 0.0
    n = len(values)
    for v in values:
        total += (v - mu) ** 2
    return total / n


def compute_accuracy(y_true, y_pred):
    correct = 0
    total = len(y_true)
    for i in range(total):
        if y_true[i] == y_pred[i]:
            correct += 1
    return correct / total

# Part2 Gaussian Naive Bayes (Abalone)

### Load Dataset

In [4]:
df = pd.read_csv("abalone.csv")
df

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


### Convert Rings to age class

In [5]:
def rings_to_class(r):
    if r <= 8:
        return "Young"
    elif r <= 11:
        return "Adult"
    else:
        return "Old"

df["AgeClass"] = df["Rings"].apply(rings_to_class)

### Select Features

In [6]:
features = [
    "Length", "Diameter", "Height",
    "Whole weight", "Shucked weight",
    "Viscera weight", "Shell weight"
]

X = df[features].values.tolist()
y = df["AgeClass"].tolist()

### Split dataset (80% training / 20% testing)

In [7]:
data = list(zip(X, y))
random.shuffle(data)

split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

X_train = [d[0] for d in train_data]
y_train = [d[1] for d in train_data]
X_test  = [d[0] for d in test_data]
y_test  = [d[1] for d in test_data]

### Compute class priors


In [8]:
def compute_priors(labels):
    priors = {}
    total = len(labels)
    for c in labels:
        priors[c] = priors.get(c, 0) + 1
    for c in priors:
        priors[c] /= total
    return priors

priors = compute_priors(y_train)

### Compute mean and variance for each feature per class

In [9]:
def summarize_by_class(X, y):
    separated = {}
    for i in range(len(y)):
        label = y[i]
        features = X[i]
        if label not in separated:
            separated[label] = []
        separated[label].append(features)

    summaries = {}
    for label, rows in separated.items():
        summaries[label] = []
        num_features = len(rows[0])
        for j in range(num_features):
            col = [row[j] for row in rows]
            mu = compute_mean(col)
            var = compute_variance(col)
            summaries[label].append((mu, var))
    return summaries

summaries = summarize_by_class(X_train, y_train)

### Gaussian probability density function

In [10]:
def gaussian_pdf(x, mean, var):
    eps = 1e-9
    coeff = 1.0 / math.sqrt(2 * math.pi * var + eps)
    exponent = math.exp(-((x - mean) ** 2) / (2 * var + eps))
    return coeff * exponent

In [11]:
def predict_prob(sample, priors, summaries):
    probs = {}
    for label in priors:
        probs[label] = math.log(priors[label])
        for i in range(len(sample)):
            mu, var = summaries[label][i]
            p = gaussian_pdf(sample[i], mu, var)
            probs[label] += math.log(p + 1e-9)
    return probs

### Predict test samples

In [12]:
def predict(sample, priors, summaries):
    probs = predict_prob(sample, priors, summaries)
    return max(probs, key=probs.get)

In [13]:
y_pred = []
for sample in X_test:
    label = predict(sample, priors, summaries)
    y_pred.append(label)

### Compute accuracy

In [14]:
acc = compute_accuracy(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.5813397129186603


# Part3 multinomail Naive Bayes ( IMDB Movie Reviews)

### Read Text Files


In [ ]:
train_dir = "aclImdb/train"
train_texts = []  
train_labels = []  

for label in ['pos', 'neg']:
    folder = os.path.join(train_dir, label)
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
                text = f.read()
                train_texts.append(text) 
                train_labels.append(1 if label == 'pos' else 0)

print("Number of training reviews:", len(train_texts))
print("First review (raw, first 200 chars):", train_texts[0][:200], "...")
print("First review label:", train_labels[0])  

Number of training reviews: 25000
First review (raw, first 200 chars): Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's  ...
First review label: 1


In [37]:
test_dir = "aclImdb/test"

test_texts = []  
test_labels = []  

for label in ['pos', 'neg']:
    folder = os.path.join(test_dir, label)
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
                text = f.read()
                test_texts.append(text)
                test_labels.append(1 if label == 'pos' else 0)

print("\nNumber of test reviews:", len(test_texts))
print("First test review label:", test_labels[0])


Number of test reviews: 25000
First test review label: 1


###  Text preprocessing (lowercase, remove punctuation, tokenize)

In [ ]:
def text_preprocessing(text):
    text = text.lower() 
    text = text.translate(str.maketrans('', '', string.punctuation))  
    tokens = text.split() 
    return tokens

train_texts_tokens = [] 
for text in train_texts:
    tokens = text_preprocessing(text)  
    train_texts_tokens.append(tokens) 
print("First 20 tokens (train):", train_texts_tokens[0][:20])

test_texts_tokens = [] 
for text in test_texts:
    tokens = text_preprocessing(text)  
    test_texts_tokens.append(tokens) 
print("First 20 tokens (test):", test_texts_tokens[0][:20])

First 20 tokens (train): ['bromwell', 'high', 'is', 'a', 'cartoon', 'comedy', 'it', 'ran', 'at', 'the', 'same', 'time', 'as', 'some', 'other', 'programs', 'about', 'school', 'life', 'such']
First 20 tokens (test): ['i', 'went', 'and', 'saw', 'this', 'movie', 'last', 'night', 'after', 'being', 'coaxed', 'to', 'by', 'a', 'few', 'friends', 'of', 'mine', 'ill', 'admit']


### Build vocabulary from training data

In [27]:
def build_vocab(reviews_tokens):
    vocab_set = set() 
    for review in reviews_tokens:
        for word in review:
            vocab_set.add(word)
    
    vocab = list(vocab_set) 
    return vocab
vocab = build_vocab(train_texts_tokens)

print("Number of unique words in vocabulary:", len(vocab))
print("First 20 words in vocabulary:", vocab[:20])

Number of unique words in vocabulary: 121358
First 20 words in vocabulary: ['lisping', 'caliberwith', 'existentbr', 'jaq', 'yolandas', 'sohail', 'have', 'videobr', 'ultrarealism', 'unmasked', 'suri', 'cassavets', 'gravediggers', 'mayeda', 'catharsic', 'stripwhat', 'majidi', 'poeticbr', 'commenced', 'hewittdressed']


### Create bag-of-words representation

In [ ]:
def bag_of_words(reviews_tokens, vocab=None, print_sample=True):
    bow_sparse = []
    total_words = 0
    vocab_set = set(vocab) if vocab else None

    for review in reviews_tokens:
        word_count = {} 
        for word in review:
            if vocab_set and word not in vocab_set:
                continue
            word_count[word] = word_count.get(word, 0) + 1
        bow_sparse.append(word_count)
        total_words += sum(word_count.values())

    if print_sample:
        print("Number of reviews:", len(bow_sparse))
        for i in range(min(3, len(bow_sparse))):
            print(f"Review {i+1} unique words:", len(bow_sparse[i]))
        print("Total words in dataset:", total_words)

    return bow_sparse, total_words

In [47]:
bow_sparse, total_words = bag_of_words(train_texts_tokens)

Number of reviews: 25000
Review 1 unique words: 91
Review 2 unique words: 234
Review 3 unique words: 112
Total words in dataset: 5821360


### Compute class priors

In [39]:
def compute_class_priors(labels, print_result=True):
    num_pos = sum(labels)  # لأن pos = 1
    num_neg = len(labels) - num_pos
    total = len(labels)
    prior_pos = num_pos / total
    prior_neg = num_neg / total

    if print_result:
        print("Number of positive reviews:", num_pos)
        print("Number of negative reviews:", num_neg)
        print("Class prior probability for positive:", prior_pos)
        print("Class prior probability for negative:", prior_neg)

    return num_pos, num_neg, prior_pos, prior_neg

In [40]:
num_pos, num_neg, prior_pos, prior_neg = compute_class_priors(train_labels)

Number of positive reviews: 12500
Number of negative reviews: 12500
Class prior probability for positive: 0.5
Class prior probability for negative: 0.5


### Compute word probabilities using Laplace smoothing

In [ ]:
def compute_word_probabilities(bow_sparse, labels, vocab):
    word_counts_pos = {}
    word_counts_neg = {}
    total_words_pos = 0
    total_words_neg = 0
    
    for review, label in zip(bow_sparse, labels):
        if label == 1:  # Positive
            for word, count in review.items():
                word_counts_pos[word] = word_counts_pos.get(word, 0) + count
                total_words_pos += count
        else:  
            for word, count in review.items():
                word_counts_neg[word] = word_counts_neg.get(word, 0) + count
                total_words_neg += count

    V = len(vocab)
    cond_prob_pos = {}
    cond_prob_neg = {}

    for word in vocab:
        count_pos = word_counts_pos.get(word, 0)
        count_neg = word_counts_neg.get(word, 0)
        cond_prob_pos[word] = (count_pos + 1) / (total_words_pos + V)
        cond_prob_neg[word] = (count_neg + 1) / (total_words_neg + V)

    return cond_prob_pos, cond_prob_neg, total_words_pos, total_words_neg

### Predict sentiment of new reviews

In [ ]:
def predict_review(review_text, vocab, cond_prob_pos, cond_prob_neg, prior_pos, prior_neg):
    tokens = text_preprocessing(review_text)
    
    word_count_new = {}
    for word in tokens:
        if word in vocab:  
            word_count_new[word] = word_count_new.get(word, 0) + 1

    log_prob_pos = math.log(prior_pos)
    log_prob_neg = math.log(prior_neg)

    for word, count in word_count_new.items():
        prob_pos = cond_prob_pos.get(word, 1e-9)
        prob_neg = cond_prob_neg.get(word, 1e-9)
        log_prob_pos += count * math.log(prob_pos)
        log_prob_neg += count * math.log(prob_neg)
    
    predicted_label = 'pos' if log_prob_pos > log_prob_neg else 'neg'
    return predicted_label

In [50]:
cond_prob_pos, cond_prob_neg, total_words_pos, total_words_neg = compute_word_probabilities(bow_sparse, train_labels, vocab)

In [59]:
vocab = set(vocab) 
test_predictions = []
for i, review_text in enumerate(test_texts):
    pred = predict_review(review_text, vocab, cond_prob_pos, cond_prob_neg, prior_pos, prior_neg)
    test_predictions.append(1 if pred == 'pos' else 0)

In [60]:
acc = compute_accuracy(test_labels, test_predictions)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 81.54%
